<a href="https://colab.research.google.com/github/Noors-lab/Model-s_summaries/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 655.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.8 MB/s eta 0:00:00


# version 4

In [ ]:
import json
import copy
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

DATA_DIR = '/content/drive/MyDrive/VigIQ/final_dataset_v3'
SAVE_DIR = '/content/drive/MyDrive/VigIQ/v4'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{DATA_DIR}/train.json') as f:
    train_data = json.load(f)
with open(f'{DATA_DIR}/val.json') as f:
    val_data = json.load(f)
with open(f'{DATA_DIR}/test.json') as f:
    test_data = json.load(f)

X_mean = np.load(f'{DATA_DIR}/X_mean_v3.npy')
X_std  = np.load(f'{DATA_DIR}/X_std_v3.npy')

# ---------- Dataset / DataLoader ----------
class SeqDataset(Dataset):
    def __init__(self, data, mean, std):
        self.data, self.mean, self.std = data, mean, std
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        kp = np.array(item['keypoints'], dtype=np.float32)
        kp = (kp - self.mean) / self.std
        return torch.tensor(kp, dtype=torch.float32), float(item['label'])

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in seqs])
    padded = pad_sequence(seqs, batch_first=True)
    return padded, lengths, torch.tensor(labels, dtype=torch.float32)

train_loader = DataLoader(SeqDataset(train_data, X_mean, X_std), batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(SeqDataset(val_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(SeqDataset(test_data, X_mean, X_std), batch_size=64, shuffle=False, collate_fn=collate_fn)

# ---------- Model ----------
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.classifier(h_final).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ShopliftingLSTM().to(device)

n_normal = sum(1 for d in train_data if d['label'] == 0)
n_not_normal = sum(1 for d in train_data if d['label'] == 1)
pos_weight = torch.tensor([n_normal / n_not_normal]).to(device)
print(f"pos_weight = {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# ---------- Train ----------
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, lengths, labels_b in loader:
            x, labels_b = x.to(device), labels_b.to(device)
            logits = model(x, lengths)
            loss = criterion(logits, labels_b)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            all_preds.append(torch.sigmoid(logits).detach().cpu())
            all_labels.append(labels_b.cpu())
    return total_loss / len(loader.dataset), torch.cat(all_preds), torch.cat(all_labels)

best_val_loss, best_state, patience, patience_ctr = float('inf'), None, 8, 0
for epoch in range(50):
    train_loss, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_preds, val_labels_t = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss, best_state, patience_ctr = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        patience_ctr += 1
    print(f"Epoch {epoch+1:2d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | best {best_val_loss:.4f}")
    if patience_ctr >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_state)
torch.save(model.state_dict(), f'{SAVE_DIR}/model_v4.pth')
np.save(f'{SAVE_DIR}/X_mean_v4.npy', X_mean)
np.save(f'{SAVE_DIR}/X_std_v4.npy', X_std)
print(f"\nSaved v4 model to {SAVE_DIR} (val_loss={best_val_loss:.4f})")

# ---------- Threshold sweep ----------
_, val_probs, val_labels_np = run_epoch(val_loader, train=False)
val_probs, val_labels_np = val_probs.numpy(), val_labels_np.numpy()

print(f"\n{'Threshold':>10} | {'Normal-Acc':>10} | {'NotNormal-Recall':>17} | {'Flagged (n)':>11}")
print("-" * 56)
for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    preds = (val_probs >= t).astype(int)
    normal_mask, not_normal_mask = val_labels_np == 0, val_labels_np == 1
    normal_acc = (preds[normal_mask] == 0).mean()
    not_normal_recall = (preds[not_normal_mask] == 1).mean()
    print(f"{t:>10.2f} | {normal_acc*100:>9.2f}% | {not_normal_recall*100:>16.2f}% | {int(preds.sum()):>11d}")

# ---------- Source-shortcut diagnostic ----------
val_sources = [d['source'] for d in val_data]
val_prefixes = [s.split(':')[0] for s in val_sources]
val_prefix_arr = np.array(val_prefixes)

print(f"\n--- Source-shortcut check ---")
for prefix in sorted(set(val_prefixes)):
    mask = val_prefix_arr == prefix
    for true_label, name in [(0, 'NORMAL'), (1, 'NOT-NORMAL')]:
        sub_mask = mask & (val_labels_np == true_label)
        n = sub_mask.sum()
        if n > 0:
            print(f"{prefix:<20} | true={name:<10} | n={n:>4} | mean_prob={val_probs[sub_mask].mean():.4f}")

pos_weight = 1.23
Epoch  1 | train_loss 0.7506 | val_loss 0.8461 | best 0.8461
Epoch  2 | train_loss 0.7561 | val_loss 0.7870 | best 0.7870
Epoch  3 | train_loss 0.7438 | val_loss 0.7696 | best 0.7696
Epoch  4 | train_loss 0.7395 | val_loss 0.7677 | best 0.7677
Epoch  5 | train_loss 0.7348 | val_loss 0.7558 | best 0.7558
Epoch  6 | train_loss 0.7148 | val_loss 0.7512 | best 0.7512
Epoch  7 | train_loss 0.7164 | val_loss 0.7470 | best 0.7470
Epoch  8 | train_loss 0.6978 | val_loss 0.7240 | best 0.7240
Epoch  9 | train_loss 0.6960 | val_loss 0.7076 | best 0.7076
Epoch 10 | train_loss 0.7065 | val_loss 0.7685 | best 0.7076
Epoch 11 | train_loss 0.7337 | val_loss 0.7498 | best 0.7076
Epoch 12 | train_loss 0.7168 | val_loss 0.7966 | best 0.7076
Epoch 13 | train_loss 0.7219 | val_loss 0.7496 | best 0.7076
Epoch 14 | train_loss 0.7099 | val_loss 0.7455 | best 0.7076
Epoch 15 | train_loss 0.6779 | val_loss 0.7082 | best 0.7076
Epoch 16 | train_loss 0.6590 | val_loss 0.7194 | best 0.7076
Epoch 

In [ ]:
#Split	     Total	       NORMAL	      NOT-NORMAL
#Train	     1,353	       745	        608
#Validation	 290	         149	        141
#Test	       291	         163	        128

# trial cell

In [ ]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
import torch.nn as nn

SAVE_DIR = '/content/drive/MyDrive/VigIQ/v4'

# ---- Just change this filename for each new clip they send ----
CLIP_NAME = 'yt_test1.mp4'
VIDEO_PATH = f'/content/drive/MyDrive/VigIQ/{CLIP_NAME}'

# ---- Model definition (needed every fresh session) ----
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.classifier(h_final).squeeze(-1)

LEFT_HIP, RIGHT_HIP = 11, 12
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6

yolo_model = YOLO('yolo11n-pose.pt')

def normalize_keypoints(kpts_xy):
    hip_mid = (kpts_xy[LEFT_HIP] + kpts_xy[RIGHT_HIP]) / 2.0
    shoulder_mid = (kpts_xy[LEFT_SHOULDER] + kpts_xy[RIGHT_SHOULDER]) / 2.0
    scale = np.linalg.norm(shoulder_mid - hip_mid)
    if scale < 1e-6:
        scale = 1.0
    return (kpts_xy - hip_mid) / scale

def extract_video_keypoints(video_path, max_frames=50):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []
    frame_indices = list(range(total_frames)) if total_frames <= max_frames else np.linspace(0, total_frames - 1, max_frames, dtype=int)
    frames_kpts = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        results = yolo_model(frame, conf=0.15, verbose=False)
        if len(results) > 0 and results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
            kpts = results[0].keypoints.xy[0].cpu().numpy()
            if kpts.shape[0] == 17:
                frames_kpts.append(normalize_keypoints(kpts).flatten().tolist())
    cap.release()
    return frames_kpts

print(f"Extracting keypoints from {VIDEO_PATH} ...")
kp_sequence = extract_video_keypoints(VIDEO_PATH, max_frames=50)
print(f"Extracted {len(kp_sequence)} valid frames")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_mean = np.load(f'{SAVE_DIR}/X_mean_v4.npy')
X_std  = np.load(f'{SAVE_DIR}/X_std_v4.npy')

model = ShopliftingLSTM().to(device)
model.load_state_dict(torch.load(f'{SAVE_DIR}/model_v4.pth', map_location=device))
model.eval()

if len(kp_sequence) == 0:
    print("No valid frames extracted — check video path / YOLO detection.")
else:
    kp = np.array(kp_sequence, dtype=np.float32)
    kp = (kp - X_mean) / X_std
    x = torch.tensor(kp, dtype=torch.float32).unsqueeze(0).to(device)
    length = torch.tensor([kp.shape[0]])
    with torch.no_grad():
        prob = torch.sigmoid(model(x, length)).item()

    print(f"\n{CLIP_NAME} — v4")
    print(f"Predicted probability (NOT-NORMAL): {prob:.4f}")
    for t in [0.80, 0.85, 0.90, 0.95]:
        print(f"  At threshold {t}: {'FLAGGED' if prob >= t else 'normal'}")

Extracting keypoints from /content/drive/MyDrive/VigIQ/yt_test1.mp4 ...
Extracted 50 valid frames

yt_test1.mp4 — v4
Predicted probability (NOT-NORMAL): 0.3221
  At threshold 0.8: normal
  At threshold 0.85: normal
  At threshold 0.9: normal
  At threshold 0.95: normal


# multiple video trial

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO

SAVE_DIR = '/content/drive/MyDrive/VigIQ/v4'
VIDEO_DIR = '/content/drive/MyDrive/VigIQ'

CLIP_NAMES = [
    'low_res.mp4',
    'high_res1.mp4',
    'high_res.mp4'
]

# ---- Model definition (needed every fresh session) ----
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256, num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.classifier(h_final).squeeze(-1)

LEFT_HIP, RIGHT_HIP = 11, 12
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6

yolo_model = YOLO('yolo11n-pose.pt')

def normalize_keypoints(kpts_xy):
    hip_mid = (kpts_xy[LEFT_HIP] + kpts_xy[RIGHT_HIP]) / 2.0
    shoulder_mid = (kpts_xy[LEFT_SHOULDER] + kpts_xy[RIGHT_SHOULDER]) / 2.0
    scale = np.linalg.norm(shoulder_mid - hip_mid)
    if scale < 1e-6:
        scale = 1.0
    return (kpts_xy - hip_mid) / scale

def extract_video_keypoints(video_path, max_frames=50):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []
    frame_indices = list(range(total_frames)) if total_frames <= max_frames else np.linspace(0, total_frames - 1, max_frames, dtype=int)
    frames_kpts = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        results = yolo_model(frame, conf=0.15, verbose=False)
        if len(results) > 0 and results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:
            kpts = results[0].keypoints.xy[0].cpu().numpy()
            if kpts.shape[0] == 17:
                frames_kpts.append(normalize_keypoints(kpts).flatten().tolist())
    cap.release()
    return frames_kpts

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_mean = np.load(f'{SAVE_DIR}/X_mean_v4.npy')
X_std  = np.load(f'{SAVE_DIR}/X_std_v4.npy')

model = ShopliftingLSTM().to(device)
model.load_state_dict(torch.load(f'{SAVE_DIR}/model_v4.pth', map_location=device))
model.eval()

results_summary = []

for clip_name in CLIP_NAMES:
    video_path = f'{VIDEO_DIR}/{clip_name}'
    print(f"\n{'='*60}")
    print(f"Processing: {clip_name}")
    kp_sequence = extract_video_keypoints(video_path, max_frames=50)

    if len(kp_sequence) == 0:
        print(f"  No valid frames extracted — skipping.")
        results_summary.append((clip_name, None, None, None))
        continue

    kp = np.array(kp_sequence, dtype=np.float32)
    kp = (kp - X_mean) / X_std
    x = torch.tensor(kp, dtype=torch.float32).unsqueeze(0).to(device)
    length = torch.tensor([kp.shape[0]])
    with torch.no_grad():
        prob = torch.sigmoid(model(x, length)).item()

    verdict_80 = 'FLAGGED' if prob >= 0.80 else 'normal'
    verdict_90 = 'FLAGGED' if prob >= 0.90 else 'normal'

    print(f"  Frames used: {len(kp_sequence)}")
    print(f"  Probability (NOT-NORMAL): {prob:.4f}")
    print(f"  At 0.80: {verdict_80}")
    print(f"  At 0.90: {verdict_90}")

    results_summary.append((clip_name, prob, verdict_80, verdict_90))

print(f"\n\n{'='*60}")
print("SUMMARY — ALL CLIPS")
print(f"{'='*60}")
print(f"{'Clip':<28} | {'Prob':>6} | {'@0.80':>8} | {'@0.90':>8}")
print("-" * 60)
for name, prob, v80, v90 in results_summary:
    if prob is None:
        print(f"{name:<28} | {'N/A':>6} | {'skipped':>8} | {'skipped':>8}")
    else:
        print(f"{name:<28} | {prob:>6.4f} | {v80:>8} | {v90:>8}")


Processing: low_res.mp4
  Frames used: 29
  Probability (NOT-NORMAL): 0.3520
  At 0.80: normal
  At 0.90: normal

Processing: high_res1.mp4
  Frames used: 33
  Probability (NOT-NORMAL): 0.1970
  At 0.80: normal
  At 0.90: normal

Processing: high_res.mp4
  Frames used: 26
  Probability (NOT-NORMAL): 0.5605
  At 0.80: normal
  At 0.90: normal


SUMMARY — ALL CLIPS
Clip                         |   Prob |    @0.80 |    @0.90
------------------------------------------------------------
low_res.mp4                  | 0.3520 |   normal |   normal
high_res1.mp4                | 0.1970 |   normal |   normal
high_res.mp4                 | 0.5605 |   normal |   normal
